# Robustness Training Notebook

This notebook trains separate QLoRA adapters to study instruction-following robustness under prompt corruption. It is set up to run on a local machine or workstation and saves all generated artifacts to local directories.

The workflow is:
1. set up local paths and check the Python environment
2. load and clean the Alpaca dataset
3. build deterministic train/validation/test splits
4. generate noisy prompt variants
5. train one adapter per corruption condition
6. save adapters, split files, and run metadata locally

## Local setup and how to run

This notebook is intended for a local Python environment rather than Google Colab.

### Hardware and access requirements
- Linux or WSL with an NVIDIA GPU and CUDA support
- access to `meta-llama/Llama-3.2-3B-Instruct` on Hugging Face
- enough disk space for model weights, checkpoints, and CSV outputs

The 4-bit QLoRA setup used here depends on `bitsandbytes`, so it is not designed for Apple Silicon or CPU-only execution.

### Recommended terminal setup

```bash
python -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install transformers datasets peft bitsandbytes trl accelerate pandas scikit-learn nltk jupyter ipykernel
python -m ipykernel install --user --name robustness-training
```

### Environment variables

Set these before launching Jupyter:

```bash
export HF_TOKEN="your_huggingface_token"
export ROBUSTNESS_PROJECT_DIR="$HOME/llama_robustness_project/v1_experiment"
export HF_HOME="$HOME/.cache/huggingface"
# Optional:
export MODEL_ID="meta-llama/Llama-3.2-3B-Instruct"
```

`ROBUSTNESS_PROJECT_DIR` is where this notebook will save:
- `data_splits/`
- `trained_adapters/`
- `training_variants_metadata.csv`

### Running the notebook

```bash
jupyter lab
```

Then open this notebook, select the `robustness-training` kernel if needed, and run the cells from top to bottom.

In [ ]:
from pathlib import Path
import importlib.util
import os

PROJECT_ROOT = Path(
    os.environ.get("ROBUSTNESS_PROJECT_DIR", "./llama_robustness_project/v1_experiment")
).expanduser().resolve()
SPLIT_DIR = PROJECT_ROOT / "data_splits"
ADAPTER_DIR = PROJECT_ROOT / "trained_adapters"
HF_CACHE_DIR = Path(
    os.environ.get("HF_HOME", PROJECT_ROOT / ".hf_cache")
).expanduser().resolve()

MODEL_ID = os.environ.get("MODEL_ID", "meta-llama/Llama-3.2-3B-Instruct")
HF_TOKEN = os.environ.get("HF_TOKEN")

for directory in [PROJECT_ROOT, SPLIT_DIR, ADAPTER_DIR, HF_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
save_path = str(PROJECT_ROOT)  # kept for compatibility with later cells
model_id = MODEL_ID

print("Project root:", PROJECT_ROOT)
print("Split directory:", SPLIT_DIR)
print("Adapter directory:", ADAPTER_DIR)
print("Hugging Face cache:", HF_CACHE_DIR)
print("Model ID:", MODEL_ID)

if HF_TOKEN:
    print("HF_TOKEN detected in the environment.")
else:
    print("HF_TOKEN is not set. If the model is gated, model loading will fail until the token is provided.")

## Check the local Python environment

The next cell verifies that the required packages are installed in the current kernel. This avoids modifying the environment from inside the notebook and makes the setup easier to reproduce.

In [ ]:
required_packages = [
    "transformers",
    "datasets",
    "peft",
    "bitsandbytes",
    "trl",
    "accelerate",
    "pandas",
    "sklearn",
    "nltk",
]

missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    raise ImportError(
        "Missing packages in the active environment: "
        + ", ".join(missing_packages)
        + "\nInstall them in your local environment before running the rest of the notebook."
    )

print("All required packages are available in the current environment.")

## Load the source dataset and perform initial cleaning

This step loads the cleaned Alpaca dataset and applies basic text cleanup before stricter filtering. The goal is to remove obviously unusable rows early so the downstream splits are built from a more consistent pool of examples.

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

print("Loading dataset...")
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
df = dataset.to_pandas()

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text

print(f"Original row count: {len(df)}")

df["instruction"] = df["instruction"].apply(normalize_text)
df["input"] = df.get("input", pd.Series([""] * len(df))).apply(normalize_text)
df["output"] = df["output"].apply(normalize_text)

placeholders = ["", "n/a", "none", "null"]
df = df[~df["output"].str.lower().isin(placeholders)]

df = df.drop_duplicates(subset=["instruction", "input"])

print(f"Row count after initial cleaning: {len(df)}")

## Authenticate to Hugging Face

This notebook reads the Hugging Face token from the `HF_TOKEN` environment variable. Keeping authentication outside the notebook avoids hardcoding credentials in the submitted file.

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("Skipping explicit login because HF_TOKEN is not set.")
    print("If the model is gated and not already cached, set HF_TOKEN and rerun this cell.")

## Build the filtered dataset and save deterministic splits

This section applies the project-specific filtering rules, removes sequences that are too long for the training setup, and creates fixed train/validation/test splits. The CSV files are saved locally so the same examples can be reused later for evaluation.

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import re

def is_valid_task(text):
    text_lower = str(text).lower()

    visual_keywords = ["generate an image", "draw a", "logo", "paint a", "picture of"]
    if any(keyword in text_lower for keyword in visual_keywords):
        return False

    math_keywords = ["calculate", "arithmetic", "equation", "integral", "derivative", "convert"]
    if re.search(r"\d+\s*[\+\-\*\/=]\s*\d+", text_lower):
        return False
    if any(keyword in text_lower for keyword in math_keywords):
        return False

    return True

print("Applying task and math filtering...")
df = df[df["instruction"].apply(is_valid_task)]
print(f"Row count after task/math filtering: {len(df)}")

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN if HF_TOKEN else None)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def get_token_length(row):
    full_text = f"{row['instruction']} {row['input']} {row['output']}"
    return len(tokenizer.encode(full_text))

print("Calculating token lengths...")
df["token_length"] = df.apply(get_token_length, axis=1)

df = df[df["token_length"] < 512]
print(f"Row count after length filtering (< 512 tokens): {len(df)}")

total_needed = 16500
if len(df) > total_needed:
    df_sampled = df.sample(n=total_needed, random_state=42)
else:
    print("Dataset is smaller than 16,500 rows after filtering. Using all available rows.")
    df_sampled = df.sample(frac=1.0, random_state=42)

train_temp, test_clean_df = train_test_split(
    df_sampled, test_size=(500 / 16500), random_state=42
)
train_df, val_df = train_test_split(
    train_temp, test_size=(1000 / 16000), random_state=42
)

print(
    f"Final splits - train: {len(train_df)}, validation: {len(val_df)}, "
    f"test-clean: {len(test_clean_df)}"
)

train_df.reset_index(drop=True).to_csv(SPLIT_DIR / "train_split.csv", index=False)
val_df.reset_index(drop=True).to_csv(SPLIT_DIR / "val_split.csv", index=False)
test_clean_df.reset_index(drop=True).to_csv(SPLIT_DIR / "test_clean_split.csv", index=False)

print(f"Saved deterministic data splits to: {SPLIT_DIR}")

## Define the prompt-corruption functions

The next cell builds the synthetic noise operators used in the study. It also creates a deterministic noisy version of the clean test set and saves that file locally. Using seeded corruption makes the noisy evaluation prompts reproducible across runs.

In [ ]:
import random
import string
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def inject_typo(text, rng=None):
    """Character-level insert/delete/substitute/swap on non-stopwords."""
    rng = rng or random
    if not text:
        return text

    words = text.split()
    if not words:
        return text

    candidates = [i for i, word in enumerate(words) if word.lower() not in stop_words and len(word) > 3]
    if not candidates:
        candidates = list(range(len(words)))

    idx = rng.choice(candidates)
    word = list(words[idx])

    if len(word) > 1:
        op = rng.choice(["insert", "delete", "substitute", "swap"])
        char_idx = rng.randint(0, len(word) - 1)

        if op == "insert":
            word.insert(char_idx, rng.choice(string.ascii_lowercase))
        elif op == "delete" and len(word) > 2:
            word.pop(char_idx)
        elif op == "substitute":
            word[char_idx] = rng.choice(string.ascii_lowercase)
        elif op == "swap" and char_idx < len(word) - 1:
            word[char_idx], word[char_idx + 1] = word[char_idx + 1], word[char_idx]

    words[idx] = "".join(word)
    return " ".join(words)

def remove_punctuation(text, rng=None):
    """Delete common punctuation marks."""
    if not text:
        return text
    chars_to_remove = ".,?:;\"'"
    translator = str.maketrans("", "", chars_to_remove)
    return text.translate(translator)

def drop_words(text, rng=None):
    """Remove a short token span or a few low-content words."""
    rng = rng or random
    if not text:
        return text

    words = text.split()
    if len(words) <= 3:
        return text

    drop_count = rng.randint(1, min(2, len(words) - 1))
    start_idx = rng.randint(0, len(words) - drop_count)
    del words[start_idx:start_idx + drop_count]
    return " ".join(words)

def shorten_prompt(text, rng=None):
    """Remove common polite wrappers or filler phrases."""
    if not text:
        return text

    fillers = [
        r"(?i)^please\s+",
        r"(?i)^can you\s+",
        r"(?i)^i would like you to\s+",
        r"(?i)^could you please\s+",
        r"(?i)please$",
    ]
    for filler in fillers:
        text = re.sub(filler, "", text).strip()
    return text

NOISE_OPERATORS = {
    "typos": inject_typo,
    "punctuation": remove_punctuation,
    "word_drop": drop_words,
    "shorthand": shorten_prompt,
}

def apply_noise(text, profile="mixed", rng=None):
    """Apply either a specific corruption or a mixed 1-2 operator corruption."""
    rng = rng or random
    if not text:
        return text

    if profile != "mixed":
        return NOISE_OPERATORS[profile](text, rng=rng)

    transforms = list(NOISE_OPERATORS.values())
    num_ops = rng.choice([1, 2])
    chosen_ops = rng.sample(transforms, num_ops)

    noisy_text = text
    for op in chosen_ops:
        noisy_text = op(noisy_text, rng=rng)

    return noisy_text

print("Generating paired test-noisy dataset...")
test_noisy_df = test_clean_df.copy()

def make_deterministic_noise(text, seed_offset):
    local_rng = random.Random(42 + seed_offset)
    return apply_noise(text, profile="mixed", rng=local_rng)

test_noisy_df["instruction"] = [
    make_deterministic_noise(text, i) for i, text in enumerate(test_noisy_df["instruction"].tolist())
]
test_noisy_df["input"] = [
    make_deterministic_noise(text, 10_000 + i) if text else text
    for i, text in enumerate(test_noisy_df["input"].tolist())
]

test_noisy_df.reset_index(drop=True).to_csv(SPLIT_DIR / "test_noisy_split.csv", index=False)

print(f"Created {len(test_noisy_df)} noisy evaluation prompts.")
print(f"Saved noisy test split to: {SPLIT_DIR / 'test_noisy_split.csv'}")
print("\nExample clean prompt:")
print(test_clean_df.iloc[0]["instruction"])
print("\nExample noisy prompt:")
print(test_noisy_df.iloc[0]["instruction"])

## Configure the base model and LoRA settings

The configuration is defined once here, but a fresh model is loaded inside the training loop for each condition. That keeps the corruption conditions independent and avoids carrying learned weights from one run into the next.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device was detected. This notebook uses 4-bit QLoRA with bitsandbytes and requires an NVIDIA GPU."
    )

use_bf16 = torch.cuda.is_bf16_supported()
torch_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"Configuring QLoRA for {model_id}...")
print(f"Using dtype: {torch_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

def build_fresh_model():
    """Load a fresh quantized base model and attach a fresh LoRA adapter."""
    print("Loading a fresh quantized base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch_dtype,
        token=HF_TOKEN if HF_TOKEN else None,
    )

    base_model = prepare_model_for_kbit_training(base_model)
    peft_model = get_peft_model(base_model, peft_config)
    peft_model.config.use_cache = False

    print("Trainable parameter summary:")
    peft_model.print_trainable_parameters()
    return peft_model

print("Model configuration is ready. A fresh adapter will be created for each training run.")

## Format prompts and train one adapter per condition

This is the main training section. For each corruption setting, the notebook:
- resets the random seed
- builds a fresh quantized base model with a fresh LoRA adapter
- formats the training text with the requested corruption level
- keeps validation prompts clean for checkpoint selection
- saves the best adapter checkpoint and a metadata table locally

If `IS_TEST_RUN` is set to `True`, the notebook uses small subsets and a short run so the pipeline can be checked quickly.

In [ ]:
import gc
import numpy as np
import pandas as pd
import random
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

IS_TEST_RUN = False
BASE_SEED = 42
RUN_SINGLE_NOISE_ABLATIONS = False

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def format_prompt(row, noise_prob=0.0, noise_profile="mixed"):
    instruction = row["instruction"]
    input_text = row["input"]
    output = row["output"]

    if noise_prob > 0 and random.random() < noise_prob:
        instruction = apply_noise(instruction, profile=noise_profile)
        if input_text:
            input_text = apply_noise(input_text, profile=noise_profile)

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    return {"text": prompt}

def format_clean_prompt(row):
    instruction = row["instruction"]
    input_text = row["input"]
    output = row["output"]

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    return {"text": prompt}

mixed_variants = [
    {
        "label": f"{int(rho * 100)}% mixed noise",
        "save_name": f"llama3-3b-adapter-noise-{int(rho * 100)}",
        "noise_prob": rho,
        "noise_profile": "mixed",
        "seed": BASE_SEED + idx,
    }
    for idx, rho in enumerate([0.0, 0.25, 0.5, 0.75])
]

single_noise_variants = [
    {"label": "50% typo-only noise", "save_name": "llama3-3b-adapter-typo-only", "noise_prob": 0.5, "noise_profile": "typos", "seed": BASE_SEED + 100},
    {"label": "50% punctuation-only noise", "save_name": "llama3-3b-adapter-punctuation-only", "noise_prob": 0.5, "noise_profile": "punctuation", "seed": BASE_SEED + 101},
    {"label": "50% shorthand-only noise", "save_name": "llama3-3b-adapter-shorthand-only", "noise_prob": 0.5, "noise_profile": "shorthand", "seed": BASE_SEED + 102},
    {"label": "50% word-drop-only noise", "save_name": "llama3-3b-adapter-word-drop-only", "noise_prob": 0.5, "noise_profile": "word_drop", "seed": BASE_SEED + 103},
]

training_variants = mixed_variants + (single_noise_variants if RUN_SINGLE_NOISE_ABLATIONS else [])

common_training_args = {
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "num_train_epochs": 3,
    "bf16": use_bf16,
    "fp16": not use_bf16,
    "optim": "paged_adamw_32bit",
    "report_to": "none",
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_total_limit": 1,
    "gradient_checkpointing": False,
    "max_grad_norm": 0.3,
    "dataset_text_field": "text",
    "max_length": 512,
}

if IS_TEST_RUN:
    print("Running a short test pass.")
    common_training_args["max_steps"] = 10
    common_training_args["logging_steps"] = 5
else:
    print("Running the full training schedule.")
    common_training_args["logging_steps"] = 50

metadata_rows = []

for variant in training_variants:
    print("\n" + "=" * 60)
    print(f"Starting training run: {variant['label']}")
    print("=" * 60)

    set_global_seed(variant["seed"])
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    current_train_df = train_df.head(40).copy() if IS_TEST_RUN else train_df.copy()
    current_val_df = val_df.head(20).copy() if IS_TEST_RUN else val_df.copy()

    formatted_train_texts = current_train_df.apply(
        lambda row: format_prompt(
            row,
            noise_prob=variant["noise_prob"],
            noise_profile=variant["noise_profile"],
        )["text"],
        axis=1,
    )
    formatted_val_texts = current_val_df.apply(
        lambda row: format_clean_prompt(row)["text"],
        axis=1,
    )

    train_dataset = Dataset.from_dict({"text": formatted_train_texts.tolist()})
    eval_dataset = Dataset.from_dict({"text": formatted_val_texts.tolist()})

    output_dir = ADAPTER_DIR / variant["save_name"]
    temp_results_dir = output_dir / "trainer_state"
    output_dir.mkdir(parents=True, exist_ok=True)
    temp_results_dir.mkdir(parents=True, exist_ok=True)

    model = build_fresh_model()
    training_args = SFTConfig(
        output_dir=str(temp_results_dir),
        seed=variant["seed"],
        **common_training_args,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
    )

    trainer.train()

    print(f"Training complete for {variant['label']}. Saving best adapter checkpoint...")
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    metadata_rows.append(
        {
            "label": variant["label"],
            "save_name": variant["save_name"],
            "noise_prob": variant["noise_prob"],
            "noise_profile": variant["noise_profile"],
            "seed": variant["seed"],
            "num_train_epochs": common_training_args["num_train_epochs"],
            "selection_metric": common_training_args["metric_for_best_model"],
            "validation_set": "clean validation split",
            "output_dir": str(output_dir),
        }
    )

    del trainer
    del model
    del train_dataset
    del eval_dataset
    del formatted_train_texts
    del formatted_val_texts
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata_rows)
metadata_path = PROJECT_ROOT / "training_variants_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)

print(f"Saved training metadata to: {metadata_path}")
print("All training runs completed.")

## Inspect the saved outputs

The final cell lists the main files and folders created by the notebook. This makes it easy to confirm that the run wrote everything to the local project directory instead of a cloud-mounted path.

In [ ]:
from pathlib import Path

print("Saved project contents:")
for item in sorted(PROJECT_ROOT.iterdir()):
    print("-", item)

if (PROJECT_ROOT / "training_variants_metadata.csv").exists():
    print("\nMetadata preview:")
    display(pd.read_csv(PROJECT_ROOT / "training_variants_metadata.csv").head())